# Day 27 — **Secure Coding: Input/Output Validation**

**Phase 1 · Module 4: Agentic System Design Patterns** · ~30 min

The syllabus says *"Run Bandit on your codebase: `bandit -r . -ll`"*. **Bandit** is a static analyser that walks your code's **AST** looking for insecure patterns — `eval`, `subprocess(..., shell=True)`, hardcoded passwords, weak crypto. To understand *how* it works (and to run keyless, without installing it), we'll build a **mini-Bandit** with the standard-library `ast` module, then add the input-validation helpers every agent boundary needs.

Today:
1. A sample module riddled with classic issues.
2. An `ast.NodeVisitor` that flags dangerous calls — the core Bandit mechanic.
3. A rule for hardcoded secrets.
4. Input-validation helpers (whitelist over blacklist).

## 1. A sample module to scan

Real Bandit reads files; we'll scan a string of source. It contains three planted issues: `eval` on input, `shell=True`, and a hardcoded API key.

In [1]:
SAMPLE = '''
import subprocess

API_KEY = "sk-live-abcdef1234567890"          # hardcoded secret

def run(cmd, expr):
    subprocess.call(cmd, shell=True)          # shell injection risk
    return eval(expr)                          # arbitrary code execution
'''
print(SAMPLE)


import subprocess

API_KEY = "sk-live-abcdef1234567890"          # hardcoded secret

def run(cmd, expr):
    subprocess.call(cmd, shell=True)          # shell injection risk
    return eval(expr)                          # arbitrary code execution



☝ To a human these jump out; at scale (thousands of files, many hands) they slip through review. Static analysis catches them **mechanically, every commit** — which is why `bandit -r .` belongs in CI, not in a once-a-quarter manual audit.

## 2. `ast.NodeVisitor` — flag dangerous calls

Parse the source to a tree, then visit every `Call` node. If the called name is on a blocklist (or `subprocess` is called with `shell=True`), record a finding with its line number — exactly what Bandit reports.

In [2]:
import ast

DANGEROUS_CALLS = {"eval", "exec", "compile", "__import__"}

class MiniBandit(ast.NodeVisitor):
    def __init__(self):
        self.findings = []

    def _name(self, node):
        if isinstance(node, ast.Name): return node.id
        if isinstance(node, ast.Attribute): return node.attr
        return None

    def visit_Call(self, node):
        fn = self._name(node.func)
        if fn in DANGEROUS_CALLS:
            self.findings.append((node.lineno, "B001", f"dangerous call: {fn}()"))
        # subprocess(..., shell=True)
        for kw in node.keywords:
            if kw.arg == "shell" and getattr(kw.value, "value", False) is True:
                self.findings.append((node.lineno, "B602", "subprocess with shell=True"))
        self.generic_visit(node)

def scan(src):
    v = MiniBandit(); v.visit(ast.parse(src)); return v.findings

for line, code_, msg in scan(SAMPLE):
    print(f"L{line}: [{code_}] {msg}")

L7: [B602] subprocess with shell=True
L8: [B001] dangerous call: eval()


☝ Two findings, each with a line number and rule id — the same shape Bandit emits. The engine is just **AST + a visitor**: parse once, walk the `Call` nodes, match against a rule set. `generic_visit` recurses so nested calls aren't missed. This is the whole idea behind every Python linter and security scanner.

## 3. A hardcoded-secrets rule

Add a second pass over **assignments**: if a variable named like a secret (`key`, `password`, `token`, `secret`) is set to a string literal, flag it. This is Bandit's `B105`/`B106` family.

In [3]:
import re
SECRET_RE = re.compile(r"(api[_-]?key|password|passwd|secret|token)", re.IGNORECASE)

def scan_secrets(src):
    findings = []
    for node in ast.walk(ast.parse(src)):
        if isinstance(node, ast.Assign):
            for tgt in node.targets:
                name = getattr(tgt, "id", "")
                if SECRET_RE.search(name) and isinstance(node.value, ast.Constant) \
                        and isinstance(node.value.value, str):
                    findings.append((node.lineno, "B105", f"hardcoded secret in '{name}'"))
    return findings

for line, code_, msg in scan_secrets(SAMPLE):
    print(f"L{line}: [{code_}] {msg}")

L4: [B105] hardcoded secret in 'API_KEY'


☝ `ast.walk` flattens the tree so we don't need a full visitor for a simple rule. It catches `API_KEY = "sk-live-..."` — a leaked credential that would otherwise sit in git history forever. The fix in real code: read secrets from the environment or a secrets manager, never a literal. A scanner that runs pre-commit stops the leak *before* it's pushed.

## 4. Input validation — whitelist, don't blacklist

Scanning finds bad code; validation stops bad **data**. The rule for agent boundaries: define what's *allowed* and reject everything else. A whitelist can't be bypassed by an input you didn't think to blacklist.

In [4]:
def validate_account(acc: str) -> str:
    if not re.fullmatch(r"AC-\d{4}", acc):        # allow ONLY this exact shape
        raise ValueError(f"invalid account id: {acc!r}")
    return acc

def validate_amount(x) -> float:
    amt = float(x)
    if not (0 < amt <= 1_000_000):
        raise ValueError(f"amount out of range: {amt}")
    return round(amt, 2)

print("ok:", validate_account("AC-1001"), validate_amount("250.5"))
for bad in [lambda: validate_account("AC-1001; DROP TABLE"),
            lambda: validate_amount(-5)]:
    try: bad()
    except ValueError as e: print("rejected:", e)

ok: AC-1001 250.5
rejected: invalid account id: 'AC-1001; DROP TABLE'
rejected: amount out of range: -5.0


☝ `re.fullmatch` with an exact pattern means `"AC-1001; DROP TABLE"` never gets through — there's no clever payload that also matches `AC-\d{4}`. Whitelisting flips the security burden: instead of enumerating infinite bad inputs, you enumerate the finite good ones. Combine with the output scrubbing from today's LangGraph lesson and both boundaries of the agent are validated.

## 5. Recap — make insecurity fail the build

| Tool | Mechanism | Catches |
|---|---|---|
| `ast.parse` + `NodeVisitor` | walk the syntax tree | `eval`/`exec`, `shell=True` (how Bandit works) |
| `ast.walk` + name match | scan assignments | hardcoded secrets |
| `re.fullmatch` whitelist | allow-list validation | injection / out-of-range inputs |
| `bandit -r . -ll` (real tool) | 100+ built-in rules | the above, plus weak crypto, unsafe yaml, … |

We rebuilt Bandit's core in ~20 lines to see there's no magic: **it's AST pattern-matching**. In real projects run the actual `bandit -r . -ll` in CI (installs via `uv add --dev bandit`) and pair it with whitelist validation at every input. Static analysis + input validation is defence *before* runtime — the cheapest security you can buy.